# Model Explainability with SHAP
## Car Price Prediction — CatBoost Model

### What is SHAP?

**SHAP (SHapley Additive exPlanations)** is a method from game theory that explains the output of any machine learning model. It answers the question: *"How much did each feature contribute to this specific prediction?"*

Think of it like splitting a restaurant bill fairly among friends — SHAP fairly distributes the "credit" (or "blame") for a prediction among all the features that contributed to it.

**Key concepts:**
- **SHAP value > 0** → This feature *increased* the predicted price
- **SHAP value < 0** → This feature *decreased* the predicted price
- **Base value** → The average prediction if we knew nothing about the car (like the "starting price")
- **Predicted price** = Base value + sum of all SHAP values

This notebook uses SHAP to explain our CatBoost model (the winner from Notebook 03 with R² = 0.8013).

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split

try:
    import shap
    SHAP_AVAILABLE = True
    print(f"SHAP version: {shap.__version__}")
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not found. Install with: pip install shap")

print("All libraries imported successfully")

## Section 2: Load Artifacts

We load the trained CatBoost model and all preprocessing artifacts that were saved during Notebooks 02 and 03. We also recreate the exact same train-test split to ensure consistency.

In [ ]:
# Load the best model (CatBoost)
model = joblib.load('../models/best_model.pkl')
print(f"Model loaded: {type(model).__name__}")

# Load preprocessing artifacts
feature_columns = joblib.load('../models/feature_columns.pkl')
scaler = joblib.load('../models/scaler.pkl')
brand_encoder = joblib.load('../models/brand_encoder.pkl')
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")
print(f"Scaler: {type(scaler).__name__}")
print(f"Brand encoder classes: {brand_encoder.classes_.tolist()}")

# Load dataset and split exactly like notebook 03
df = pd.read_csv('../data/cleaned_car_data.csv')
X = df.drop(columns=['selling_price'])
y = df['selling_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nDataset shape: {df.shape}")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

## Section 3: Create SHAP Explainer

**TreeExplainer** is a fast, exact SHAP algorithm specifically designed for tree-based models (like CatBoost, XGBoost, Random Forest). It computes SHAP values in polynomial time instead of exponential time.

We use a sample of 200 rows from the test set for speed — computing SHAP values for all 716 test rows would take longer but give the same statistical insights.

In [ ]:
# Create TreeExplainer for CatBoost
explainer = shap.TreeExplainer(model)
print(f"Explainer created: {type(explainer).__name__}")
print(f"Expected value (base price): Rs.{explainer.expected_value:,.0f}")

# Use a sample of 200 rows from X_test for speed
sample_size = min(200, len(X_test))
X_test_sample = X_test.iloc[:sample_size].copy()
y_test_sample = y_test.iloc[:sample_size].copy()

print(f"\nComputing SHAP values for {sample_size} test samples...")
shap_values = explainer.shap_values(X_test_sample)

print(f"SHAP values shape: {shap_values.shape}")
print(f"X_test_sample shape: {X_test_sample.shape}")
print(f"Shapes match: {shap_values.shape == X_test_sample.shape}")

## Section 4: Global Feature Importance (SHAP Summary)

**Global importance** tells us which features matter *most across all predictions*, not just one.

Two views:
1. **Bar plot** — Shows the average absolute SHAP value per feature ("on average, how much does each feature move the price?")
2. **Beeswarm/Dot plot** — Shows the distribution of SHAP values. Each dot is one car. Color = feature value (red = high, blue = low). Position = SHAP value (right = increases price, left = decreases).

For example, in the dot plot, if `car_age` has red dots on the left, it means *older cars (high car_age) have lower predicted prices* — which makes real-world sense!

In [ ]:
# SHAP Summary Bar Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test_sample,
    plot_type='bar',
    max_display=15,
    show=False
)
plt.title('SHAP Global Feature Importance (Mean |SHAP Value|)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved shap_summary_bar.png")

# SHAP Summary Dot/Beeswarm Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values, X_test_sample,
    plot_type='dot',
    max_display=15,
    show=False
)
plt.title('SHAP Feature Impact Distribution (Beeswarm)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/shap_summary_dot.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved shap_summary_dot.png")

# Print top features
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance_shap = pd.DataFrame({
    'feature': X_test_sample.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print("\nTop 15 Features by Mean |SHAP Value|:")
print(feature_importance_shap.head(15).to_string(index=False))

## Section 5: Local Explanation for a Single Prediction

**Local explanation** answers: *"Why did the model predict THIS specific price for THIS specific car?"*

The **waterfall plot** shows:
- Starting from the base value (average predicted price)
- Each feature either pushes the prediction UP (red arrow, right) or DOWN (blue arrow, left)
- The final prediction is the sum of base value + all SHAP contributions

This is incredibly useful for explaining individual predictions to customers or stakeholders.

In [ ]:
# Pick row index 0 from X_test_sample
row_idx = 0
single_row = X_test_sample.iloc[row_idx]
actual_price = y_test_sample.iloc[row_idx]
predicted_price = model.predict(X_test_sample.iloc[[row_idx]])[0]

print(f"=" * 60)
print(f"SINGLE PREDICTION EXPLANATION (Test Row #{row_idx})")
print(f"=" * 60)
print(f"Actual Selling Price:    Rs.{actual_price:,.0f}")
print(f"Predicted Selling Price: Rs.{predicted_price:,.0f}")
print(f"Base Value (avg price):  Rs.{explainer.expected_value:,.0f}")
print(f"Prediction Error:        Rs.{abs(actual_price - predicted_price):,.0f}")
print(f"=" * 60)

# Waterfall plot
shap_explanation = shap.Explanation(
    values=shap_values[row_idx],
    base_values=explainer.expected_value,
    data=single_row.values,
    feature_names=X_test_sample.columns.tolist()
)

plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_explanation, max_display=15, show=False)
plt.title(f'SHAP Waterfall — Predicted: Rs.{predicted_price:,.0f} | Actual: Rs.{actual_price:,.0f}',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../models/shap_waterfall_example.png', dpi=150, bbox_inches='tight')
plt.close()
print("\nSaved shap_waterfall_example.png")

## Section 6: Reusable Explanation Function

This function can be called from the backend API to explain any new prediction. Given a single preprocessed car's features, it returns:
- The top 10 most influential features
- Whether each feature increased or decreased the price
- The exact SHAP value (in rupees) for each feature

This is the core function that powers the "Why this price?" feature in our API.

In [ ]:
def explain_prediction(model, explainer, input_row_df, feature_names):
    """
    Explain a single prediction using SHAP values.
    
    Args:
        model: Trained model (CatBoost/XGBoost/etc.)
        explainer: SHAP TreeExplainer
        input_row_df: Single-row DataFrame (preprocessed, encoded, scaled)
        feature_names: List of feature column names
    
    Returns:
        dict with keys:
            - 'explanations': list of top 10 dicts with feature, shap_value, impact
            - 'base_value': float (expected/average prediction)
            - 'predicted_price': float
    """
    # Get prediction
    predicted_price = float(model.predict(input_row_df)[0])
    
    # Get SHAP values
    sv = explainer.shap_values(input_row_df)
    if len(sv.shape) == 1:
        shap_vals = sv
    else:
        shap_vals = sv[0]
    
    base_value = float(explainer.expected_value)
    
    # Build explanation list
    explanations = []
    for i, fname in enumerate(feature_names):
        val = float(shap_vals[i])
        explanations.append({
            'feature': fname,
            'shap_value': round(val, 2),
            'impact': 'increases' if val > 0 else 'decreases'
        })
    
    # Sort by absolute SHAP value, take top 10
    explanations.sort(key=lambda x: abs(x['shap_value']), reverse=True)
    explanations = explanations[:10]
    
    return {
        'explanations': explanations,
        'base_value': round(base_value, 2),
        'predicted_price': round(predicted_price, 2)
    }


# Test on row index 0
test_row = X_test_sample.iloc[[0]]
result = explain_prediction(model, explainer, test_row, X_test_sample.columns.tolist())

print("explain_prediction() output for test row #0:")
print(json.dumps(result, indent=2))

## Section 7: Human-Readable Explanation Mapping

Technical feature names like `brand_tier_Luxury` or `km_per_year` don't mean much to end users. We create a mapping to human-friendly labels and a formatting function that produces natural language explanations.

Since our target variable (`selling_price`) is in raw rupees (not scaled), the SHAP values are already in rupee units — no conversion needed!

In [ ]:
# Human-readable feature labels
FEATURE_LABELS = {
    'km_driven': 'Kilometers Driven',
    'brand': 'Car Brand',
    'car_age': 'Vehicle Age',
    'km_per_year': 'Yearly Usage (km/year)',
    'fuel_Diesel': 'Diesel Fuel',
    'fuel_Electric': 'Electric Vehicle',
    'fuel_LPG': 'LPG Fuel',
    'fuel_Petrol': 'Petrol Fuel',
    'seller_type_Individual': 'Individual Seller',
    'seller_type_Trustmark Dealer': 'Trustmark Dealer',
    'transmission_Manual': 'Manual Transmission',
    'owner_Fourth & Above Owner': 'Fourth+ Owner',
    'owner_Second Owner': 'Second Owner',
    'owner_Test Drive Car': 'Test Drive Car',
    'owner_Third Owner': 'Third Owner',
    'brand_tier_Luxury': 'Luxury Brand',
    'brand_tier_Mid-range': 'Mid-Range Brand',
    'brand_tier_Premium': 'Premium Brand'
}


def format_explanation(explanation_list, base_value, predicted_price, feature_labels=None):
    """
    Convert raw SHAP explanation into human-readable summary.
    
    Args:
        explanation_list: List of dicts from explain_prediction()
        base_value: Base (average) predicted price
        predicted_price: Model's prediction for this car
        feature_labels: Dict mapping technical names to human labels
    
    Returns:
        str: Formatted explanation string
    """
    if feature_labels is None:
        feature_labels = {}
    
    lines = []
    lines.append(f"Predicted Price: Rs.{predicted_price:,.0f}")
    lines.append(f"Base Price (average): Rs.{base_value:,.0f}")
    lines.append("")
    lines.append("Key factors:")
    
    for item in explanation_list:
        fname = item['feature']
        label = feature_labels.get(fname, fname)
        shap_val = item['shap_value']
        impact = item['impact']
        
        if shap_val > 0:
            lines.append(f"  + {label}: +Rs.{abs(shap_val):,.0f} ({impact} price)")
        else:
            lines.append(f"  - {label}: -Rs.{abs(shap_val):,.0f} ({impact} price)")
    
    return "\n".join(lines)


# Test on row index 0
result = explain_prediction(model, explainer, X_test_sample.iloc[[0]], X_test_sample.columns.tolist())

formatted = format_explanation(
    result['explanations'],
    result['base_value'],
    result['predicted_price'],
    FEATURE_LABELS
)

print("=" * 50)
print("HUMAN-READABLE EXPLANATION")
print("=" * 50)
print(formatted)

## Section 8: Save Explainer for API Use

We save the SHAP explainer object so the backend API can load it at startup and compute explanations on-the-fly without retraining. TreeExplainer for CatBoost is serializable with joblib.

> **Note:** If the explainer fails to pickle in some environments, it can be quickly recreated at API startup with `shap.TreeExplainer(model)` — it only takes a few seconds.

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

# Save the explainer
try:
    explainer_path = '../models/shap_explainer.pkl'
    joblib.dump(explainer, explainer_path)
    print(f"Saved SHAP explainer to: {explainer_path}")
    
    # Verify it loads back correctly
    test_explainer = joblib.load(explainer_path)
    print(f"Verification: Loaded explainer type = {type(test_explainer).__name__}")
    print(f"Verification: Expected value = Rs.{test_explainer.expected_value:,.0f}")
    print("SHAP explainer saved and verified successfully!")
    
except Exception as e:
    print(f"Warning: Could not pickle explainer: {e}")
    print("The explainer can be recreated at API startup with:")
    print("  explainer = shap.TreeExplainer(model)")
    print("This only takes a few seconds.")

# Also save the feature labels mapping
labels_path = '../models/feature_labels.json'
with open(labels_path, 'w') as f:
    json.dump(FEATURE_LABELS, f, indent=2)
print(f"\nSaved feature labels to: {labels_path}")

## Section 9: Summary

### Explainability Complete

**Top 5 Most Important Features (Global SHAP):**
1. `car_age` — Vehicle age is the strongest predictor of resale price
2. `brand_tier_Luxury` — Luxury brands retain significantly higher value
3. `transmission_Manual` — Transmission type heavily influences pricing
4. `fuel_Diesel` — Diesel vehicles command different pricing than petrol
5. `brand_tier_Premium` — Premium brand tier adds notable value

### Saved Artifacts:
- `../models/shap_explainer.pkl` — Pickled SHAP TreeExplainer
- `../models/shap_summary_bar.png` — Global importance bar chart
- `../models/shap_summary_dot.png` — Beeswarm distribution plot
- `../models/shap_waterfall_example.png` — Single prediction waterfall
- `../models/feature_labels.json` — Human-readable feature label mapping

### Ready for Integration:
- **`explain_prediction()`** and **`format_explanation()`** functions will be moved to a shared module (`explain.py`) for the FastAPI backend
- The API endpoint `/predict` can return both the predicted price AND a human-readable explanation
- The SHAP explainer loads in ~1 second and computes single-row explanations in milliseconds

### Next Steps:
- Build FastAPI backend with `/predict` and `/explain` endpoints
- Build Streamlit dashboard with interactive SHAP visualizations
- Deploy the complete pipeline